In [ ]:
# Generate Kilosort channel map (.mat) files from an Open Ephys recording folder.
# One .mat per ephys probe stream (handles single- and multi-probe recordings),
# each saved next to structure.oebin as "<recording name>_<stream>.mat".
# Existing files are overwritten (overwrite=True in the last cell).

In [1]:
# Channel indices to exclude from sorting (dead / noise / out-of-brain).
# 0-indexed, same numbering as OpenEphys.
#
# Multi-probe recording: use a dict keyed by stream name (missing stream -> none
# excluded). Single probe: a dict with one key, or just a plain list (applied to
# every probe).
bad_channels = {
    "ProbeA": [6, 25, 32, 90, 121, 137, 251],
    "ProbeB": [0, 2, 10, 54, 58, 131, 218, 240],
}

In [ ]:
# path and params
from pathlib import Path
base_path = Path(r"D:\D1-1-6_IM-1979\ephys\2026-09-04_14-20-43")

In [3]:
import sys
# from IPython.core.getipython import get_ipython
# sys.path.append(str(Path(__file__).parent.parent) if "__file__" in dir() else "..")

sys.path.append("..")
from src import oe_parse_folders, oe_parse_params, save_kilosort_channel_map

oe_names, oe_paths = oe_parse_folders(base_path)
probes_params, _ = oe_parse_params(oe_paths["xml"], oe_paths["oebin"])


def bad_for(stream_name):
    """Resolve the exclusion list for one probe stream (dict per stream, or a
    single list shared by all)."""
    if isinstance(bad_channels, dict):
        return bad_channels.get(stream_name, [])
    return bad_channels


print(f"Recording : {base_path.name}")
print(f"Probes found: {len(probes_params)}")
for p in probes_params:
    print(f"  {p['stream_name']}  —  {p['probe_name']}  ({p['channel_count']} ch, "
          f"{len(bad_for(p['stream_name']))} channels excluded)")

Recording : 2026-09-04_14-20-43
Probes found: 2
  ProbeA  —  Neuropixels 2.0 - Multishank  (384 ch, 7 channels excluded)
  ProbeB  —  Neuropixels 2.0 - Multishank  (384 ch, 8 channels excluded)


In [4]:
# generate channel map for each ephys probe (one .mat per stream, next to structure.oebin)
for probe_params in probes_params:
    save_kilosort_channel_map(probe_params, oe_paths["oebin"].parent, base_path.name,
                              bad_channels=bad_for(probe_params["stream_name"]),
                              overwrite=True)

Saved 2026-09-04_14-20-43_ProbeA.mat  (377 channels, 7 excluded)
Saved 2026-09-04_14-20-43_ProbeB.mat  (376 channels, 8 excluded)
